In [1]:
import pandas as pd
import numpy as np

In [25]:
ptm_var = pd.read_csv("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/labeled_data/PTMVar.csv",skiprows = 6).iloc[:,:26]

In [26]:
ptm_var = ptm_var.loc[ptm_var["MOD_TYPE"] == "Phosphorylation",:]

In [27]:
ptm_var.groupby("VAR_TYPE").count().iloc[:,[0]]

,GENE
VAR_TYPE,
Disease,2295
LB/B,10790
LP/P,7449
US,3518


In [29]:
ptm_var = ptm_var.loc[:,["GENE","UPID","WT_AA","MUT_RSD#","VAR_AA","VAR_TYPE","VAR_CLASS","VAR_POSITION"]]

In [37]:
ptm_var["VAR_POSITION"] = ptm_var["VAR_POSITION"].apply(lambda s : int(s[1:]))

In [38]:
ptm_var["p_ind"] = ptm_var["MUT_RSD#"]-ptm_var["VAR_POSITION"]

In [39]:
ptm_var.columns = ["substrate_gene","substrate","wt_aa","mut_ind","mut_aa","var_type","var_class","var_dist","p_ind"]

In [54]:
ptm_var.shape

(24052, 9)

In [186]:
ptm_var = ptm_var.reset_index().iloc[:,1:].reset_index()

### Match ptm_var to known ksi

In [187]:
import os
DATA_DIR = "${SPARK_ACQUISITION_ROOT}/kinase_ml/data"
train_dat = pd.read_pickle(os.path.join(DATA_DIR,"interactions","interaction_classed_221017.pkl"))


In [188]:
train_dat.loc[train_dat["substrate"]=="Q9NQ94",:]

,substrate,kinase,p_ind,p_aa,substrate_seq,kinase_seq,kinase_pfam,kinase_interpro,domain_cc,domain_ft,protein_families,kinase_flag,kinase_coord,valid_check,superfamily,pk_family,pk_subfamily


In [189]:
ptm_ksi = pd.merge(ptm_var,train_dat,on = ["substrate","p_ind"])


In [191]:
ptm_unmapped = ptm_var.loc[~ptm_var["index"].isin(ptm_ksi["index"]),:]

In [192]:
ptm_ksi = ptm_ksi.iloc[:,1:]
ptm_ksi.shape

(4265, 24)

In [ ]:
wt_train_dat = ptm_ksi.copy()

In [86]:
ptm_ksi.groupby("var_type").count().iloc[:,0]

var_type
Disease     259
LB/B        709
LP/P       1246
US         2051
Name: substrate_gene, dtype: int64

In [87]:
mut_train_dat = ptm_ksi.copy()

In [88]:
mut_substrate_seq = []
for i in range(mut_train_dat.shape[0]):
    s = mut_train_dat.iloc[i,:]
    mut_subseq = s["substrate_seq"]
    list1 = list(mut_subseq)
    assert mut_subseq[s["mut_ind"]-1]==s["wt_aa"]
    list1[int(s["mut_ind"])-1] = s["mut_aa"]
    mut_subseq = ''.join(list1)
    mut_substrate_seq.append(mut_subseq)

In [89]:
mut_train_dat["substrate_seq"] = mut_substrate_seq

In [90]:
wt_train_dat["wt"] = True
mut_train_dat["wt"] = False

In [91]:
train_dat_wt_mut = pd.concat([wt_train_dat,mut_train_dat]).loc[:,['substrate', 'kinase', 'p_ind', 'p_aa', 'substrate_seq', 'kinase_seq',
       'kinase_pfam', 'kinase_interpro', 'domain_cc', 'domain_ft',
       'protein_families', 'kinase_flag', 'kinase_coord', 'valid_check',
       'superfamily', 'pk_family', 'pk_subfamily', 'wt']]

In [92]:
train_dat_wt_mut.to_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/labeled_data/ptm_var.pkl")

In [93]:
ptmvar_predictions = pd.read_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/predictions/tapeClassify_ptmvar_predictions_15aa_15aa.pkl")

In [103]:
ptmvar_predictions["logit"] = ptmvar_predictions["logits"].apply(lambda s: s[1])
c = ptmvar_predictions.loc[ptmvar_predictions["wt"]==True,:]
ptmvar_predictions_mut = ptmvar_predictions.loc[ptmvar_predictions["wt"]==False,:]

In [105]:
import seaborn as sns

In [110]:
ptm_ksi["wt_logit"] = ptmvar_predictions_wt["logit"]
ptm_ksi["mt_logit"] = ptmvar_predictions_mut["logit"]

In [124]:
ptm_ksi["mut_score"] =  ptm_ksi["mt_logit"] - ptm_ksi["wt_logit"]
ptm_ksi["wt_pred"] = (ptm_ksi["wt_logit"]>0)
ptm_ksi["mt_pred"] = (ptm_ksi["mt_logit"]>0)
# ptm_ksi_wous = ptm_ksi.loc[(ptm_ksi["var_type"]!='US')&(ptm_ksi["var_type"]!='Disease'),:]

In [174]:
thresh = 0
ptm_ksi_stats = pd.merge(ptm_ksi.groupby("var_type").count().iloc[:,[0]].reset_index(),ptm_ksi.loc[ptm_ksi["mut_score"]<thresh,:].groupby("var_type").count().iloc[:,[0]].reset_index(),on = "var_type")
ptm_ksi_stats.columns = ["var_type","total_n","pred_lof"]
ptm_ksi_stats["frac"] = ptm_ksi_stats["pred_lof"]/ptm_ksi_stats["total_n"]
ptm_ksi_stats

,var_type,total_n,pred_lof,frac
0,Disease,259,198,0.764479
1,LB/B,709,534,0.753173
2,LP/P,1246,988,0.792937
3,US,2051,1537,0.749391


In [180]:
thresh = 0
ptm_ksi_stats = pd.merge(ptm_ksi.groupby("var_type").count().iloc[:,[0]].reset_index(), \
                         ptm_ksi.loc[ptm_ksi["mt_logit"]<thresh,:].groupby("var_type").count().iloc[:,[0]].reset_index(), \
                         on = "var_type")
ptm_ksi_stats.columns = ["var_type","total_n","pred_lof"]
ptm_ksi_stats["frac"] = ptm_ksi_stats["pred_lof"]/ptm_ksi_stats["total_n"]
ptm_ksi_stats

,var_type,total_n,pred_lof,frac
0,Disease,259,26,0.100386
1,LB/B,709,95,0.133992
2,LP/P,1246,174,0.139647
3,US,2051,332,0.161872


1. most of the variants, in any of the category, results in substrates that still predicted to be binding. It makes sense in that were the variants to be non-interacting at all, it would be very detrimental and thus most likely resulting not a survived sample

2. most of the variants results in a decrease in logit, suggesting either a real "disturbing" effect or potentially a WT training bias



### For psites with no known kinase
1. predict against all known kinases to find the top candidates
2. perform the mutegenesis prediction agains the imputed KSIs

In [263]:
human_kinome = pd.read_csv("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/raw/human_kinome.csv")

In [261]:
# human_kinome

In [252]:
uniprot_kinase_interpro = pd.read_csv("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/raw/uniprot_kinase_interpro.tab",sep = "\t")

/usr/local/lib/python3.8/dist-packages/IPython/core/interactiveshell.py:3135: DtypeWarning: Columns (11,13) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


In [264]:
human_kinome = pd.merge(human_kinome,uniprot_kinase_interpro,left_on = "UniprotID",right_on = "Entry")

In [268]:
human_kinome['Domain [FT]']

0      DOMAIN 46..315;  /note="Protein kinase";  /evi...
1      DOMAIN 61..121;  /note="SH3";  /evidence="ECO:...
2      DOMAIN 107..167;  /note="SH3";  /evidence="ECO...
3      DOMAIN 126..385;  /note="Protein kinase";  /ev...
4      DOMAIN 192..485;  /note="Protein kinase";  /ev...
                             ...                        
507    DOMAIN 27..311;  /note="Protein kinase";  /evi...
508    DOMAIN 91..352;  /note="Protein kinase";  /evi...
509    DOMAIN 67..328;  /note="Protein kinase";  /evi...
510    DOMAIN 285..540;  /note="Protein kinase";  /ev...
511    DOMAIN 33..122;  /note="Ig-like C2-type 1"; DO...
Name: Domain [FT], Length: 512, dtype: object

In [259]:
human_kinome = human_kinome.loc[:,["gene","Kinase Name","Group","Family","SubFamily","UniprotID","Organism","Sequence",'Length']]

In [269]:
def get_kinase_domain_ind(s):
    s = str(s)
    try:
        res = [doc.split(";")[0].split("..") for doc in s.split("DOMAIN ")[1:] if "Protein kinase" in doc]
    except:
        res = []
    if len(res) == 0:
        try:
            res = [doc.split(";")[0].split("..") for doc in s.split("DOMAIN ")[1:] if "Histidine kinase" in doc]
        except:
            res = []
    if len(res) == 0:
        try:
            res = [doc.split(";")[0].split("..") for doc in s.split("DOMAIN ")[1:] if "Fam20C" in doc]
        except:
            res = []
    return res

In [270]:
kinase_coord = human_kinome["Domain [FT]"].apply(lambda s: get_kinase_domain_ind(s))

In [272]:
# kinase_coord

In [273]:
human_kinome["kinase_coord"] = kinase_coord

In [274]:
human_kinome["kinase_coord"].apply(lambda s: len(s)==1).sum()

472

In [275]:
human_kinome["kinase_coord"].apply(lambda s: len(s)>1).sum()

27

In [277]:
human_kinome = human_kinome.loc[human_kinome["kinase_coord"].apply(lambda s: len(s)==1),:]

In [279]:
human_kinome = human_kinome.loc[:,["gene",'Kinase Name','Group', 'Family', 'SubFamily','UniprotID','Sequence', 'Length','kinase_coord']]

In [280]:
human_kinome.columns = ['kinase_gene', 'kinase_name', 'pk_group', 'pk_family', 'pk_subfamily','kinase',"kinase_seq","kinseq_len","kinase_coord"]

In [283]:
ptm_unmapped ["substrate"].to_csv("tmp_csv_221106.csv",index = False)
# TO-DO: fetch substrate sequence and run predictions

In [285]:
sub_unmap_uniprot = pd.read_csv("./uniprot-compressed_true_download_true_fields_accession_2Creviewed_2C-2022.11.07-06.28.10.46.tsv",sep = "\t")

In [286]:
sub_unmap_uniprot.head(1)

,From,Entry,Reviewed,Entry Name,Protein names,Organism,Sequence
0,Q9NQ94,Q9NQ94,reviewed,A1CF_HUMAN,APOBEC1 complementation factor (APOBEC1-stimul...,Homo sapiens (Human),MESNHKSGDGLSGTQKEAALRALVQRTGYSLVQENGQRKYGGPPPG...


In [290]:
ptm_unmapped = pd.merge(ptm_unmapped,sub_unmap_uniprot,left_on = "substrate",right_on = "Entry").iloc[:,[1,2,3,4,5,6,7,8,9,14,15,16]]

In [292]:
ptm_unmapped.to_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/labeled_data/ptm_unmapped_substrates.pkl")

In [294]:
human_kinome.to_pickle("${SPARK_ACQUISITION_ROOT}/kinase_ml/data/labeled_data/human_kinome_seq.pkl")